 ### 用ConversationBufferMemory 嵌入记忆 (Deprecated)

In [1]:
from langchain_openai import ChatOpenAI
import os
import dotenv
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent, tool
from langchain_classic.memory import ConversationBufferMemory

dotenv.load_dotenv()

llm = ChatOpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),
    temperature=0,
    model="gpt-4o-mini",
)

prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant"),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{input}"),
        MessagesPlaceholder(variable_name="agent_scratchpad"),
    ]
)

@tool
def magic_function(input: int) -> int:
    """Applies a magic function to an input."""
    return input + 2

agent = create_tool_calling_agent(
    llm=llm,
    tools=[magic_function],
    prompt=prompt_template,
)

##加入记忆
memory = ConversationBufferMemory(
    memory_key="chat_history",   # 要和 MessagesPlaceholder 的名字一致
    return_messages=True         # 因为 prompt 这里要的是消息对象，不是纯字符串
)

agent_executor = AgentExecutor(
    agent=agent,
    tools=[magic_function],
    memory=memory,   ## 加入记忆
    verbose=True
)

# 第一轮
response1 = agent_executor.invoke({
    "input": "我叫小林，what is the value of magic_function(3)?"
})
print(response1)

# 第二轮
response2 = agent_executor.invoke({
    "input": "我叫什么名字？"
})
print(response2)

C:\Users\15350\AppData\Local\Temp\ipykernel_30640\2828742367.py:37: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(




> Entering new AgentExecutor chain...

Invoking: `magic_function` with `{'input': 3}`


5magic_function(3) 的值是 5。

> Finished chain.
{'input': '我叫小林，what is the value of magic_function(3)?', 'chat_history': [HumanMessage(content='我叫小林，what is the value of magic_function(3)?', additional_kwargs={}, response_metadata={}), AIMessage(content='magic_function(3) 的值是 5。', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])], 'output': 'magic_function(3) 的值是 5。'}


> Entering new AgentExecutor chain...
你叫小林。

> Finished chain.
{'input': '我叫什么名字？', 'chat_history': [HumanMessage(content='我叫小林，what is the value of magic_function(3)?', additional_kwargs={}, response_metadata={}), AIMessage(content='magic_function(3) 的值是 5。', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='我叫什么名字？', additional_kwargs={}, response_metadata={}), AIMessage(content='你叫小林。', additional_kwargs={}, response_metadata={}, tool_calls=[],

## 主线版本
## create_agent(...)
## LangGraph checkpointer 负责短期记忆

In [2]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain_openai import ChatOpenAI
import os
import dotenv

dotenv.load_dotenv()

llm = ChatOpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),
    temperature=0,
    model="gpt-4o-mini",
)

def magic_function(input: int) -> int:
    """Applies a magic function to an input."""
    return input + 2

# 用于保存短期会话记忆
checkpointer = InMemorySaver()

# 创建主线 agent
agent = create_agent(
    model=llm,
    tools=[magic_function],
    system_prompt="You are a helpful assistant",
    checkpointer=checkpointer,
)

# 用 thread_id 区分同一个会话
config = {
    "configurable": {
        "thread_id": "session_1"
    }
}

# 第一轮
response1 = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "我叫小林，what is the value of magic_function(3)?"}
        ]
    },
    config=config,
)
print("第一轮结果：")
print(response1)

# 第二轮：不用再手动传 chat_history，agent 会自动记住
response2 = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "我叫什么名字？"}
        ]
    },
    config=config,
)
print("第二轮结果：")
print(response2)

第一轮结果：
{'messages': [HumanMessage(content='我叫小林，what is the value of magic_function(3)?', additional_kwargs={}, response_metadata={}, id='17023c59-0ee6-4f1b-b36c-a84dd0a20873'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 66, 'total_tokens': 81, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_eb37e061ec', 'id': 'chatcmpl-DVYgjUu9yQB8wF1kcDaBPzaqRBnkB', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d9a84-b25a-7861-adee-7b28b8c6ef8e-0', tool_calls=[{'name': 'magic_function', 'args': {'input': 3}, 'id': 'call_ea2276DsoR2tTAioKlH9sSld', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_to